# Demand Forecasting Analysis

Analyze historical sales patterns to forecast future demand and better understand purchasing trends over time.

**Business Goal:** Support inventory planning and operational decision-making by predicting future product demand and identifying sales trends.


## 1. Import Libraries

Import libraries for time series analysis, forecasting, and visualization.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA

## 2. Load Dataset

Load historical sales data and review the dataset structure.

In [ ]:
# load data
df = pd.read_excel("Online Retail.xlsx")

## 3. Data Cleaning

Prepare and clean the dataset for forecasting analysis.

In [ ]:
# clean columns
df.columns = df.columns.str.strip().str.lower()

## 4. Exploratory Data Analysis (EDA)

Explore sales trends, seasonality, and time-based patterns.

In [ ]:
# keep valid rows only
df = df[(df["quantity"] > 0) & (df["unitprice"] > 0)].copy()

## 5. Feature Engineering

Prepare time-based features and aggregated metrics for forecasting.

In [ ]:
# create date and sales
df["invoicedate"] = pd.to_datetime(df["invoicedate"])
df["sales"] = df["quantity"] * df["unitprice"]
df["month"] = df["invoicedate"].dt.to_period("m").astype(str)

## 6. Demand Forecasting Model

Build and evaluate forecasting models to predict future demand.

In [ ]:
# monthly sales
monthly_sales = (
    df.groupby("month")["sales"]
    .sum()
    .reset_index()
)

## 7. Visualization & Key Insights

Visualize forecast results and summarize important findings.

In [ ]:
# convert month to datetime and sort
monthly_sales["month"] = pd.to_datetime(monthly_sales["month"])
monthly_sales = monthly_sales.sort_values("month").iloc[:-1].copy()

## 8. Business Recommendations

Provide recommendations for inventory and demand planning.

In [ ]:
# moving average forecast
monthly_sales["ma_3"] = monthly_sales["sales"].rolling(3).mean()
ma_forecast = monthly_sales["sales"].tail(3).mean()

print("moving average forecast for next month:", round(ma_forecast, 2))

## 9. Conclusion

Summarize findings and possible future improvements.

In [ ]:
# arima forecast
ts = monthly_sales.set_index("month")["sales"]
model = ARIMA(ts, order=(1, 1, 1))
model_fit = model.fit()
arima_forecast = model_fit.forecast(steps=1)

print("arima forecast for next month:", round(arima_forecast.iloc[0], 2))

In [ ]:
# next month for forecast points
next_month = monthly_sales["month"].max() + pd.DateOffset(months=1)

In [ ]:
# plot
plt.figure(figsize=(10, 5))
plt.plot(monthly_sales["month"], monthly_sales["sales"], marker="o", label="actual")
plt.plot(monthly_sales["month"], monthly_sales["ma_3"], linestyle="--", label="ma_3")
plt.scatter(next_month, ma_forecast, label="ma_forecast")
plt.scatter(next_month, arima_forecast.iloc[0], label="arima_forecast")

## 10. Export CSVs for Tableau Dashboard

Export forecasting results to CSV files for Tableau visualization.

In [ ]:
import os
os.makedirs("../outputs", exist_ok=True)

# Monthly sales with 3-month moving average
monthly_out = monthly_sales.copy()
monthly_out["month"] = monthly_out["month"].dt.strftime("%Y-%m")
monthly_out.to_csv("../outputs/monthly_sales.csv", index=False)

# Forecast comparison: MA vs ARIMA
forecast_df = pd.DataFrame({
    "month": [next_month.strftime("%Y-%m")],
    "ma_forecast": [round(ma_forecast, 2)],
    "arima_forecast": [round(arima_forecast.iloc[0], 2)]
})
forecast_df.to_csv("../outputs/forecast_comparison.csv", index=False)

print("Saved:")
print("  ../outputs/monthly_sales.csv -", len(monthly_out), "months")
print("  ../outputs/forecast_comparison.csv")
print()
print(forecast_df)